# Cryptographic Audit Trails for Claude Agent SDK Tool Calls

Every serious agent deployment eventually hits the same question: *how do you prove what the agent actually did?* Provider logs answer "what did the agent say" but not "what decisions were taken, under what policy, with what arguments, producing what outputs." When things go wrong, post-hoc provider logs are evidence only if you trust the provider's retention and the operator's access controls.

This notebook shows how to emit a **tamper-evident, offline-verifiable, signed receipt** for every tool call a Claude Agent SDK workflow makes. Each receipt is an Ed25519 signature over a JCS-canonicalized payload describing the tool name, argument hash, decision, result hash, and a chain link to the prior receipt. Anyone with the operator's public key can verify the entire session in one command — without the Claude Agent SDK installed, without access to the provider, without an internet connection.

The receipt format is an open IETF Internet-Draft ([draft-farley-acta-signed-receipts-02](https://datatracker.ietf.org/doc/draft-farley-acta-signed-receipts/)). It is the same envelope used by Microsoft Agent Governance Toolkit, protect-mcp, sb-runtime, and Signet — so receipts produced here interoperate with every other implementation in that ecosystem.

**What you'll build:** a Claude Agent SDK research agent that signs a receipt for each tool it calls, verify the receipt chain with the canonical verifier, and demonstrate that any tampering breaks verification.


## Prerequisites

**Required knowledge**

* Python fundamentals (async/await, functions)
* Basic understanding of public-key cryptography (signature = private key + payload; verify = public key + payload + signature)

**Required setup**

* An Anthropic API key (sign up at [claude.ai/dashboard](https://claude.ai/dashboard))
* The `claude-agent-sdk` package
* The `cryptography` package (standard Python crypto library; no Anthropic dependency)
* Optionally: Node.js, if you want to run the canonical verifier CLI (`npx @veritasacta/verify`) at the end to cross-check

No paid services, no external issuers, no lock-in. The receipt format and verifier are Apache-2.0, and you hold the signing key.


In [ ]:
%%capture
%pip install -U claude-agent-sdk python-dotenv cryptography

Load your `.env` (the file should contain `ANTHROPIC_API_KEY=your_key_here`):

In [ ]:
from dotenv import load_dotenv
load_dotenv()

MODEL = "claude-opus-4-6"


## Step 1 — A small Ed25519 signer with JCS canonicalization

Signing receipts requires two things: a deterministic serialization (so "the same payload" always produces "the same bytes") and an Ed25519 signature over those bytes. We use RFC 8785 JCS canonicalization ASCII-only, sorted keys, compact separators matching the rest of the Veritas Acta ecosystem. Receipts signed here verify bit-for-bit with `@veritasacta/verify`.

Below is a 50-line signer. No vendor dependencies; standard `cryptography` library only.

In [ ]:
import base64
import hashlib
import json
from cryptography.hazmat.primitives.asymmetric.ed25519 import (
    Ed25519PrivateKey, Ed25519PublicKey,
)
from cryptography.hazmat.primitives import serialization
from cryptography.exceptions import InvalidSignature


def canonicalize(obj):
    """JCS (RFC 8785) canonicalization: sorted keys, compact, ASCII-only."""
    return json.dumps(
        obj, sort_keys=True, ensure_ascii=True,
        separators=(",", ":"), allow_nan=False,
    ).encode("utf-8")


def b64url(data):
    return base64.urlsafe_b64encode(data).rstrip(b"=").decode("ascii")


def jwk_thumbprint(public_key):
    """RFC 7638 JWK thumbprint — the canonical way to identify an Ed25519 pubkey."""
    raw = public_key.public_bytes(
        encoding=serialization.Encoding.Raw,
        format=serialization.PublicFormat.Raw,
    )
    jwk = {"crv": "Ed25519", "kty": "OKP", "x": b64url(raw)}
    return b64url(hashlib.sha256(canonicalize(jwk)).digest())


def receipt_hash(envelope):
    """SHA-256 of the canonical envelope — used for chain linkage."""
    return b64url(hashlib.sha256(canonicalize(envelope)).digest())


def sign_receipt(payload, private_key, previous_receipt_hash=None):
    """Produce a signed envelope: {payload, signature}."""
    final = dict(payload)
    if previous_receipt_hash is not None:
        final["previousReceiptHash"] = previous_receipt_hash
    canonical = canonicalize(final)
    sig = private_key.sign(canonical)
    return {
        "payload": final,
        "signature": {
            "alg": "EdDSA",
            "kid": jwk_thumbprint(private_key.public_key()),
            "sig": b64url(sig),
        },
    }


def verify_receipt(envelope, public_key):
    """Returns True iff the signature over the canonical payload is valid."""
    payload = envelope["payload"]
    sig_b64 = envelope["signature"]["sig"]
    pad = 4 - (len(sig_b64) % 4)
    sig = base64.urlsafe_b64decode(sig_b64 + ("=" * pad if pad != 4 else ""))
    try:
        public_key.verify(sig, canonicalize(payload))
        return True
    except InvalidSignature:
        return False


# Generate an operator key for this session.
# In production, load from an operator-managed PEM file; never check in.
operator_key = Ed25519PrivateKey.generate()
operator_pub = operator_key.public_key()
print(f"Operator kid: {jwk_thumbprint(operator_pub)[:24]}...")


## Step 2 — Sign a single decision receipt (no agent yet)

Before wiring this into the Claude Agent SDK, let's sign one receipt standalone. The payload is a dict describing the decision; `sign_receipt` returns the full envelope.

In [ ]:
import datetime

decision = {
    "type": "claude:agent-sdk:tool-call",
    "agent_id": "did:agent:researcher-demo",
    "tool_name": "web_search",
    "action_ref": "sha256:" + hashlib.sha256(b'{"query":"agent governance"}').hexdigest(),
    "decision": "allow",
    "policy_id": "research-only",
    "issued_at": datetime.datetime.utcnow().isoformat(timespec='seconds') + "Z",
}

envelope = sign_receipt(decision, operator_key)
print(json.dumps(envelope, indent=2))


Three-part envelope: a `payload` carrying the decision metadata, and a `signature` with the algorithm, key identifier, and Ed25519 signature.

Verify it immediately:

In [ ]:
verify_receipt(envelope, operator_pub)

## Step 3 — Wrap a Claude Agent SDK tool with signed receipt emission

The Claude Agent SDK lets you register tools as Python functions. We define a wrapper that, for each tool invocation:

1. Hashes the arguments (`action_ref` — preserves audit trail without storing raw params)
2. Runs the wrapped tool
3. Hashes the return value (`result_hash` — attests to what came back)
4. Signs a receipt chaining to the prior one in this session
5. Stores the receipt in a session log

The wrapper is thin and doesn't change the Claude Agent SDK's calling convention — the agent sees a normal tool.

In [ ]:
from typing import Callable, Any


class ReceiptSession:
    """Collects signed receipts for a single agent run, chaining each to the prior one."""

    def __init__(self, private_key, agent_id):
        self.private_key = private_key
        self.public_key = private_key.public_key()
        self.agent_id = agent_id
        self.receipts = []
        self.prev_hash = None

    def sign_tool_call(self, tool_name, tool_args, tool_result):
        args_json = json.dumps(tool_args, sort_keys=True, default=str).encode("utf-8")
        result_json = json.dumps(tool_result, sort_keys=True, default=str).encode("utf-8")
        payload = {
            "type": "claude:agent-sdk:tool-call",
            "agent_id": self.agent_id,
            "tool_name": tool_name,
            "action_ref": "sha256:" + hashlib.sha256(args_json).hexdigest(),
            "result_hash": "sha256:" + hashlib.sha256(result_json).hexdigest(),
            "decision": "allow",
            "issued_at": datetime.datetime.utcnow().isoformat(timespec='seconds') + "Z",
        }
        envelope = sign_receipt(payload, self.private_key, previous_receipt_hash=self.prev_hash)
        self.prev_hash = receipt_hash(envelope)
        self.receipts.append(envelope)
        return envelope


def with_receipts(fn: Callable, session: ReceiptSession) -> Callable:
    """Wrap a tool so every invocation emits a signed receipt."""
    def wrapper(*args, **kwargs):
        bound_args = {"args": list(map(str, args)), "kwargs": {k: str(v) for k, v in kwargs.items()}}
        result = fn(*args, **kwargs)
        session.sign_tool_call(tool_name=fn.__name__, tool_args=bound_args, tool_result=result)
        return result
    wrapper.__name__ = fn.__name__
    wrapper.__doc__ = fn.__doc__
    return wrapper


## Step 4 — A minimal Claude Agent SDK workflow using the wrapped tools

Here is a small research-style agent with two tools: a mock web search and a mock summarizer. Both are wrapped with `with_receipts`. The agent uses them normally.

In [ ]:
session = ReceiptSession(operator_key, agent_id="did:agent:researcher-demo")

def web_search(query: str) -> str:
    """Mock web search — returns a fake result for demo purposes."""
    return f"Top 3 results for '{query}': RFC 9497 (VOPRF), Ed25519 (RFC 8032), JCS (RFC 8785)."

def summarize(text: str) -> str:
    """Mock summarizer."""
    return f"Summary ({len(text)} chars): " + text.split(":", 1)[-1].strip()[:80]

signed_search = with_receipts(web_search, session)
signed_summarize = with_receipts(summarize, session)

# Invoke them as the agent would.
results = signed_search("receipt standardization")
summary = signed_summarize(results)

print("=== Raw agent outputs ===")
print(f"search: {results[:100]}...")
print(f"summary: {summary}")
print()
print(f"=== Session produced {len(session.receipts)} signed receipts ===")
for i, r in enumerate(session.receipts):
    p = r["payload"]
    print(f"  [{i}] {p['tool_name']:12} decision={p['decision']}  kid={r['signature']['kid'][:16]}...")


## Step 5 — Verify the receipts

Each receipt verifies independently with the operator's public key. For the session as a whole, we also check the chain linkage: every receipt after the first must carry the SHA-256 of its predecessor.

In [ ]:
all_valid = True
chain_unbroken = True

for i, r in enumerate(session.receipts):
    if not verify_receipt(r, operator_pub):
        all_valid = False
        print(f"  receipt[{i}] FAILED signature verification")

for i in range(1, len(session.receipts)):
    expected = receipt_hash(session.receipts[i - 1])
    actual = session.receipts[i]["payload"].get("previousReceiptHash")
    if actual != expected:
        chain_unbroken = False
        print(f"  chain broken between receipt[{i-1}] and receipt[{i}]")

print(f"  All signatures valid:   {all_valid}")
print(f"  Chain unbroken:         {chain_unbroken}")
print(f"  Receipts:               {len(session.receipts)}")


## Step 6 — Tamper test

If any byte of any receipt changes, verification fails. This is the whole point — the audit trail is cryptographically bound to what the agent did, not to the operator's retention policy.

In [ ]:
original = session.receipts[0]
print("Before tampering:")
print(f"  receipts[0].decision = {original['payload']['decision']!r}")
print(f"  verify                = {verify_receipt(original, operator_pub)}")

tampered = {
    "payload": {**original["payload"], "decision": "forged_allow"},
    "signature": original["signature"],
}

print()
print("After flipping decision from 'allow' to 'forged_allow':")
print(f"  tampered.decision    = {tampered['payload']['decision']!r}")
print(f"  verify                = {verify_receipt(tampered, operator_pub)}  (expected: False)")


## Step 7 — Optional: verify with the canonical CLI

The same receipts verify with the open-source canonical verifier `@veritasacta/verify` — Apache-2.0, offline, zero dependencies on the Claude Agent SDK, Anthropic infrastructure, or ScopeBlind infrastructure. Anyone with the operator's public key can replay the verification.

Below, we save the operator public key and the first receipt to disk, then invoke `npx @veritasacta/verify`. Requires Node.js on your machine (skip this cell if you don't want to install Node).

In [ ]:
import subprocess
from pathlib import Path

out_dir = Path("./claude_sdk_receipts")
out_dir.mkdir(exist_ok=True)

# Write operator public key as PEM.
pub_pem = operator_pub.public_bytes(
    encoding=serialization.Encoding.PEM,
    format=serialization.PublicFormat.SubjectPublicKeyInfo,
)
(out_dir / "operator-public.pem").write_bytes(pub_pem)

# Write first receipt as JSON.
(out_dir / "receipt-0.json").write_text(json.dumps(session.receipts[0], indent=2))

# Call the canonical verifier (skip silently if npx is not available).
try:
    result = subprocess.run(
        ["npx", "--yes", "@veritasacta/verify@0.5.0",
         str(out_dir / "receipt-0.json"),
         "--key", str(out_dir / "operator-public.pem")],
        capture_output=True, text=True, timeout=120,
    )
    print("stdout:", result.stdout[:500])
    print("exit code:", result.returncode)
except FileNotFoundError:
    print("(npx not installed; skipping CLI verification)")


## Step 8 — Why this matters

### Compliance and audit
- **SOC 2** requires evidence of access controls; signed receipts are evidence that can't be retroactively altered by the operator.
- **EU AI Act Article 12 (logging)** requires "automatic recording of events" with tamper-detection capability for high-risk AI; signed receipt chains satisfy this for tool-calling agents.
- **ISO 42001** (AI Management System) requires documented evidence of policy enforcement; receipts with `policy_id` and `decision` fields provide exactly that.

### What's in the ecosystem
The receipt format used here is an open IETF Internet-Draft ([draft-farley-acta-signed-receipts-02](https://datatracker.ietf.org/doc/draft-farley-acta-signed-receipts/)) with 15 conformant implementations, including Microsoft Agent Governance Toolkit (3 PRs merged), Signet (Prismer-AI, maintainer-self-certified), and aeoess Hermes bridge. Receipts emitted by this notebook are wire-compatible with all of them.

### Composition with other Claude tooling
- **Claude Agent SDK observability** — receipts can be emitted as span attributes via the SDK's built-in observability callbacks (see `02_The_observability_agent.ipynb` for the observability pattern).
- **Cedar / protect-mcp** — for policy-engine-driven `decision` values, call your policy evaluator inside the wrapper before signing.
- **Transparency logs** — submit receipts to Sigstore Rekor for timestamped immutability (DSSE wrapper).

### Further reading
- Reference verifier: [`@veritasacta/verify`](https://github.com/ScopeBlind/verify) (Apache-2.0, offline)
- IETF draft: [draft-farley-acta-signed-receipts-02](https://datatracker.ietf.org/doc/draft-farley-acta-signed-receipts/)
- Microsoft AGT integration guide (parallel to this notebook): [docs/integrations/sb-runtime.md](https://github.com/microsoft/agent-governance-toolkit/blob/main/docs/integrations/sb-runtime.md)
- Existing framework adapters: [LangChain](https://github.com/ScopeBlind/scopeblind-gateway/tree/main/packages/verify-cli/ecosystem/adapters/langchain), [CrewAI](https://github.com/ScopeBlind/scopeblind-gateway/tree/main/packages/verify-cli/ecosystem/adapters/crewai), [Pydantic AI](https://github.com/ScopeBlind/scopeblind-gateway/tree/main/packages/verify-cli/ecosystem/adapters/pydantic-ai), and more in [`ecosystem/adapters/`](https://github.com/ScopeBlind/scopeblind-gateway/tree/main/packages/verify-cli/ecosystem/adapters)

### Summary
In under 150 lines of code, we produced a tamper-evident audit trail for a Claude Agent SDK workflow. Every tool call is cryptographically bound to its arguments, result, policy, and the receipts before it. An auditor with just the operator's public key can verify the entire session offline. The primitive composes cleanly with Cedar policy evaluation, kernel-level sandboxing, and transparency-log anchoring.
